In [0]:
# Install nfl_data_py and its dependencies
%pip install --no-deps nfl_data_py
%pip install appdirs fastparquet pandas

In [0]:
import nfl_data_py as nfl
import pandas as pd

In [0]:
# Import play-by-play data for 2025 season
pbp_2025 = nfl.import_pbp_data([2025])

# Display basic info
print(f"Loaded {len(pbp_2025)} plays from 2025 season")
print(f"\nDataFrame shape: {pbp_2025.shape}")
print(f"\nColumns: {list(pbp_2025.columns[:10])}...")  # Show first 10 columns

# Display first few rows
display(pbp_2025.head())

In [0]:
# Import schedule data for 2025 (includes Vegas lines, home/away, weather, rest)
schedules = nfl.import_schedules([2025])

print(f"Loaded {len(schedules)} games")
print(f"\nKey columns: {list(schedules.columns)}")
print(f"\nSample data:")
display(schedules.head())

In [0]:
# Import snap count data for 2025
snap_counts = nfl.import_snap_counts([2025])

print(f"Loaded {len(snap_counts)} player-game snap records")
print(f"\nColumns: {list(snap_counts.columns)}")
print(f"\nSample data:")
display(snap_counts.head())

In [0]:
# Since 2025 weekly stats are not yet available, calculate them from raw PBP data
print("Calculating weekly player statistics from play-by-play data...\n")

# ============================================================================
# 1. PASSER AGGREGATIONS
# ============================================================================
print("Aggregating passing stats...")
passing_plays = pbp_2025[pbp_2025['play_type'] == 'pass'].copy()

passer_stats = passing_plays.groupby(
    ['week', 'passer_player_id', 'passer_player_name'], 
    dropna=False
).agg({
    'pass_attempt': 'sum',
    'complete_pass': 'sum',
    'yards_gained': 'sum',
    'pass_touchdown': 'sum',
    'interception': 'sum'
}).reset_index()

# Rename columns for clarity
passer_stats.rename(columns={
    'passer_player_id': 'player_id',
    'passer_player_name': 'player_name',
    'pass_attempt': 'pass_attempts',
    'complete_pass': 'completions',
    'yards_gained': 'passing_yards',
    'pass_touchdown': 'passing_tds',
    'interception': 'interceptions'
}, inplace=True)

print(f"  → {len(passer_stats)} passer-week records")

# ============================================================================
# 2. RUSHER AGGREGATIONS
# ============================================================================
print("Aggregating rushing stats...")
rushing_plays = pbp_2025[pbp_2025['play_type'] == 'run'].copy()

rusher_stats = rushing_plays.groupby(
    ['week', 'rusher_player_id', 'rusher_player_name'],
    dropna=False
).agg({
    'rush_attempt': 'sum',
    'yards_gained': 'sum',
    'rush_touchdown': 'sum'
}).reset_index()

# Rename columns for clarity
rusher_stats.rename(columns={
    'rusher_player_id': 'player_id',
    'rusher_player_name': 'player_name',
    'rush_attempt': 'rush_attempts',
    'yards_gained': 'rushing_yards',
    'rush_touchdown': 'rushing_tds'
}, inplace=True)

print(f"  → {len(rusher_stats)} rusher-week records")

# ============================================================================
# 3. RECEIVER AGGREGATIONS
# ============================================================================
print("Aggregating receiving stats...")
receiving_plays = pbp_2025[
    (pbp_2025['play_type'] == 'pass') & 
    (pbp_2025['receiver_player_id'].notna())
].copy()

receiver_stats = receiving_plays.groupby(
    ['week', 'receiver_player_id', 'receiver_player_name'],
    dropna=False
).agg({
    'pass_attempt': 'sum',      # Targets = pass attempts to this receiver
    'complete_pass': 'sum',     # Receptions = completed passes to this receiver
    'yards_gained': 'sum',      # Receiving yards
    'pass_touchdown': 'sum'     # Receiving TDs
}).reset_index()

# Rename columns for clarity
receiver_stats.rename(columns={
    'receiver_player_id': 'player_id',
    'receiver_player_name': 'player_name',
    'pass_attempt': 'targets',
    'complete_pass': 'receptions',
    'yards_gained': 'receiving_yards',
    'pass_touchdown': 'receiving_tds'
}, inplace=True)

print(f"  → {len(receiver_stats)} receiver-week records")

# ============================================================================
# 4. MERGE ALL STATS INTO SINGLE DATAFRAME
# ============================================================================
print("\nMerging all stats...")

# Merge passing and rushing
weekly_stats = passer_stats.merge(
    rusher_stats, 
    on=['week', 'player_id', 'player_name'], 
    how='outer'
)

# Merge with receiving
weekly_stats = weekly_stats.merge(
    receiver_stats, 
    on=['week', 'player_id', 'player_name'], 
    how='outer'
)

# Coalesce player names (fill nulls from any of the three sources)
weekly_stats['player_name'] = weekly_stats['player_name'].fillna(method='bfill', axis=0).fillna(method='ffill', axis=0)

# Fill all numeric NaNs with 0
numeric_cols = [
    'pass_attempts', 'completions', 'passing_yards', 'passing_tds', 'interceptions',
    'rush_attempts', 'rushing_yards', 'rushing_tds',
    'targets', 'receptions', 'receiving_yards', 'receiving_tds'
]
weekly_stats[numeric_cols] = weekly_stats[numeric_cols].fillna(0)

print(f"  → {len(weekly_stats)} total player-week records")

# ============================================================================
# 5. CALCULATE FANTASY POINTS (FULL PPR)
# ============================================================================
print("\nCalculating fantasy points (PPR scoring)...")

weekly_stats['fantasy_points_ppr'] = (
    # Passing
    (weekly_stats['passing_yards'] * 0.04) +
    (weekly_stats['passing_tds'] * 4) +
    (weekly_stats['interceptions'] * -2) +
    
    # Rushing
    (weekly_stats['rushing_yards'] * 0.1) +
    (weekly_stats['rushing_tds'] * 6) +
    
    # Receiving
    (weekly_stats['receiving_yards'] * 0.1) +
    (weekly_stats['receiving_tds'] * 6) +
    (weekly_stats['receptions'] * 1)  # PPR bonus
)

# Round to 2 decimal places
weekly_stats['fantasy_points_ppr'] = weekly_stats['fantasy_points_ppr'].round(2)

# Sort by fantasy points for easy inspection
weekly_stats = weekly_stats.sort_values('fantasy_points_ppr', ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("WEEKLY STATS CALCULATION COMPLETE")
print("="*80)
print(f"Total records: {len(weekly_stats):,}")
print(f"Columns: {list(weekly_stats.columns)}")
print(f"\nTop 10 performances by fantasy points:")
display(weekly_stats.head(10))

In [0]:
# Import injury data for 2024 and 2025
injuries = nfl.import_injuries([2024, 2025])

print(f"Loaded {len(injuries)} injury records")
print(f"\nColumns: {list(injuries.columns)}")
print(f"\nSample data:")
display(injuries.head())